In [2]:
import os
%load_ext autoreload
%autoreload 1
%aimport utility.functions
%aimport src.analysis
%aimport src.dataset
%aimport src.plots
%aimport src.simulation
import numpy as np
import utility.functions as fn
import src.simulation as sim
import src.analysis as an
import src.dataset as ds
import src.plots as plots
import torch
from scipy.stats import norm
import pandas as pd
from utility.double_beta_spectrum import pdf_ratio2b
import glob
import matplotlib.pyplot as plt
BASE = "../"
channels = [3, 5, 9, 11, 13, 15, 17, 19]
channel_ID = [5, 9, 11, 10, 4, 2, 12, 3]
StdCuts = [0.0003, 0.0004, 0.0008, 0.0006, 0.0012, 0.0007, 0.00035, 0.0025]
signal_amps =  [0.028,  0.033,  0.035,  0.035,   0.04,  0.031,  0.022,  0.018,  0.015]

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Main loop to compute the BI for all channels and acceptance levels

In [17]:
window_size = 4096
meas_name = "000813_20230628T161508"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
BIs = []
BI_uncertainties = []
channel_list = []
M_list = []
M_train_list = []
acceptances_list = []
channels = [3, 5, 9, 11, 13, 15, 17, 19]
channel_ID = [5, 9, 11, 10, 4, 2, 12, 3]
StdCuts = [0.0003, 0.0004, 0.0008, 0.0006, 0.0012, 0.0007, 0.00035, 0.0025]

scaling_factor_correction = 1.

# acceptances = [0.7, 0.8, 0.84, 0.87, 0.9, 0.92, 0.93, 0.94, 0.95, 0.96, 0.99]
acceptances = [0.9]

compute_simulated_BI = False
train_m_fcts = False

test_name = "base"

for channel, StdCut in zip(channels, StdCuts):
    if compute_simulated_BI:
        #meanpulse = np.fromfile(ds.find_file(f"{meas_name}_{channel:03}_???.bin_edmean.bin",
        #                                     specific_subdir="RUN9_pulse_injected_new/"))
        #meanpulse_inj = np.fromfile(
        #    f"{BASE}/outputs/meanpulses_build/pos_pulses_rawdata/average_pulse_channel_{channel}.bin", dtype=np.float32)
        meanpulse_inj = np.load(f"{BASE}/outputs/meanpulses_build/pos_pulses_rawdata/average_pulse_channel{channel}.npy")
        # Load datasets and channel specifications
        dataset_single_name = "pup_n1-d0_000813_20230628T161508"
        file_path_single, path_pos_single = ds.find_files(meas_name, dataset_single_name, channel,
                                                          specific_subdir="RUN9_pulse_injected_new/")
        data_single = ds.CachedBinaryDataset_withgenerated(file_path_single, path_pos_single, window_size,
                                                           pulse=meanpulse_inj,
                                                           n_windows=15999, win_shift=0,
                                                           scaling_factor=scaling_factor_correction)
    
        dataset_pileup_name = "pup_n1-d8_000813_20230628T161508"
        file_path_pileup, path_pos_pileup = ds.find_files(meas_name, dataset_pileup_name, channel,
                                                          specific_subdir="RUN9_pulse_injected_new/")
        data_pileup = ds.CachedBinaryDataset_withgenerated(file_path_pileup, path_pos_pileup, window_size,
                                                           pulse=meanpulse_inj,
                                                           n_windows=15999, win_shift=0,
                                                           scaling_factor=scaling_factor_correction)
        # data_single, data_pileup = ds.create_data_sets_denoise(channel, win_length=2000, shift=0, pos_prefix="_bis")
    # load NPS 
    nps_bis = np.load(f"{BASE}/outputs/NPS_study/NPS_channel{channel}_win{window_size}_{meas_name}.npy")
    nps_bis *= (8. / 3.)  # Adjust for Hann window effect
    # load meanpulse for analysis 
    meanpulse_analysis = np.load(f"{BASE}/outputs/meanpulses_build/channel{channel}_meanpulse_mean.npy")
    meanpulse_analysis = meanpulse_analysis[np.argmax(meanpulse_analysis) - window_size // 2:
                                            np.argmax(meanpulse_analysis) + window_size // 2]

    signal_amp = ds.get_amp_Q_val(channel) * scaling_factor_correction

    # Loop over acceptance levels
   
    for acceptance in acceptances:
        N_sigma = norm.ppf(1 - (1 - acceptance) * 100 / 100)
        S, w, H_unit = an.compute_H(meanpulse_analysis, nps_bis, np.hanning)
        # Convert all arrays to torch tensors on GPU
        S_torch = torch.tensor(S, dtype=torch.cfloat, device=device)
        H_unit_torch = torch.tensor(H_unit, dtype=torch.cfloat, device=device)
        w_torch = torch.tensor(w, dtype=torch.cfloat, device=device)
        nps_torch = torch.tensor(nps_bis, dtype=torch.cfloat, device=device)

        t_min, t_max, N_t = 0, 8e-4, 100
        r_min, r_max, N_r = 0., .5, 100
        ratio_distribution = pdf_ratio2b(np.linspace(r_min, r_max, N_r))
        ratio_distribution /= np.mean(ratio_distribution)

        t_torch = torch.linspace(t_min, t_max, N_t, dtype=torch.cfloat, device=device)
        r_torch = torch.linspace(r_min, r_max, N_r, dtype=torch.cfloat, device=device)
        ratio_distribution_torch = torch.tensor(ratio_distribution, dtype=torch.cfloat, device=device)
        if train_m_fcts:
            f1_t, f2_t, M_values = an.optimize_filters(S_torch, H_unit_torch, w_torch, t_torch, r_torch, nps_torch,
                                                           signal_amp, ratio_distribution_torch, N_sigma=N_sigma,
                                                           n_trials=1000, verbose=True)
            f1_opt,f2_opt = f1_t.numpy(),f2_t.numpy()
            
            np.save(f"{BASE}/outputs/training_Ms/channel_{channel}/M_{int(acceptance * 100):d}_{test_name}", np.array(M_values))
            np.save(f"{BASE}/outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance * 100):d}_configIV",
                    np.array([f1_opt, f2_opt]))
        else:
            try:
                f1_opt, f2_opt = np.load(f"{BASE}/outputs/Pileup_filter_functions/channel_{channel}/functions_eff{int(acceptance * 100):d}_{test_name}.npy")
                M_values = np.load(f"{BASE}/outputs/training_Ms/channel_{channel}/M_{int(acceptance*100):d}_{test_name}.npy")
            except FileNotFoundError:
                raise FileNotFoundError(f"{test_name} never been trained")
            
        M_train_final = M_values[-1]
        
        M_train_list.append(float(M_train_final))
        acceptances_list.append(acceptance)
        if compute_simulated_BI:
            f1_t = torch.tensor(f1_opt, dtype = torch.float32, device = device)
            f2_t = torch.tensor(f2_opt, dtype = torch.float32, device = device)
            S_inj, w_inj, H_unit_inj = an.compute_H(meanpulse_inj[np.argmax(meanpulse_inj)-window_size//2:
                                                                  np.argmax(meanpulse_inj)+window_size//2], 
                                                    nps_bis, np.hanning)
            S_inj_torch = torch.tensor(S, dtype = torch.cfloat, device = device)
            H_unit_inj_torch = torch.tensor(H_unit, dtype = torch.cfloat, device = device)
            w_inj_torch = torch.tensor(w, dtype = torch.cfloat, device = device)
            S_H_delayed, S_H, S2_over_nps = an.precompute_constants(S_torch, H_unit_torch, w_torch, t_torch, nps_torch)
            M = an.compute_M(f1_t, f2_t, S_H_delayed, r_torch, S_H, S2_over_nps, signal_amp, ratio_distribution_torch,
                             N_sigma = N_sigma, full_output = False, use_interp = True)
            
            # Compute PSD and BI
            BI, rp, sigma_rp, sigma_BI = an.compute_BI_torch(
                data_pileup,
                data_single,
                acceptance,
                H_unit_torch,
                f1_t,
                f2_t,
                window_fct = np.hanning,
                n_deriv = 0,
                compute_uncertainty = True,
                batch_size = 2048,
                use_loader = False
            )
            BIs.append(float(BI))
            BI_uncertainties.append(float(sigma_BI))
            channel_list.append(channel)
            M_list.append(float(M))
                    
            data = np.column_stack([channel_list,acceptances_list, BIs, BI_uncertainties,M_list,M_train_list])
            header = "channel,acceptance,BI,BI_uncertainty,M,M_train"
    
            np.savetxt(
                f"{BASE}/outputs/BI_results_{test_name}.csv",
                data,
                delimiter = ",",
                header = header,
                comments = ""
            )
        else:
            data = np.column_stack([channel_list,acceptances_list, M_train_list])
            header = "channel,acceptance,M_train"
            np.savetxt(
                f"{BASE}/outputs/BI_results_{test_name}.csv",
                data,
                delimiter = ",",
                header = header,
                comments = ""
            )
        break

Final: J = 0.263186


/tmp/ipykernel_8749/2807516560.py:80: FutureWarning: The input object of type 'Tensor' is an array-like implementing one of the corresponding protocols (`__array__`, `__array_interface__` or `__array_struct__`); but not a sequence (or 0-D). In the future, this object will be coerced as if it was first converted using `np.array(obj)`. To retain the old behaviour, you have to either modify the type 'Tensor', or assign to an empty array created with `np.empty(correct_shape, dtype=object)`.
  np.array([f1_opt, f2_opt]))


ValueError: setting an array element with a sequence. The requested array has an inhomogeneous shape after 1 dimensions. The detected shape was (2,) + inhomogeneous part.

Get the SNR, M and BI for denoise vs raw plot

In [11]:
window_size = 2048
meas_name = "000813_20230628T161508"
device = 'cuda' if torch.cuda.is_available() else 'cpu'
for idx in np.argsort(channel_ID):
    if channel_ID[idx] not in [5]:
        continue
    channel = channels[idx]
    nps_raw = np.load(f"{BASE}/outputs/NPS_study/NPS_channel{channel}_win{window_size}_{meas_name}.npy")*(8./3.)

    f1_raw,f2_raw = np.load(f"{BASE}/outputs/Pileup_filter_functions/channel_{channel}/window_size/functions_eff90_window_size_2048.npy")
    
    meanpulse = np.load(f"{BASE}/outputs/meanpulses_build/channel{channel}_meanpulse_raw.npy")
    def compute_mu_sigma(save_name,f1_raw,f2_raw,nps_raw,meanpulse):
        f1_raw,f2_raw = torch.tensor(f1_raw),torch.tensor(f2_raw)
        window_size = len(nps_raw)
        meanpulse_bis = meanpulse[np.argmax(meanpulse)-window_size//2 : np.argmax(meanpulse)+window_size//2]
        
        S, w, H_unit_raw = an.compute_H(meanpulse_bis, nps_raw, np.hanning)
        nps_raw = torch.tensor(nps_raw)
        f1_raw =f1_raw/torch.mean(torch.abs(f1_raw * H_unit_raw * S)).real
        f2_raw =f2_raw/torch.mean(torch.abs(f2_raw * H_unit_raw * S)).real
        
        signal_amp = ds.get_amp_Q_val(channel)
        t_min, t_max, N_t = 0, 8e-4, 100
        
        r_min, r_max, N_r = 0., .5, 100
        ratio_distribution = pdf_ratio2b(np.linspace(r_min, r_max, N_r))
        ratio_distribution /= np.mean(ratio_distribution)
    
        t_torch = torch.linspace(t_min, t_max, N_t, dtype=torch.cfloat, device=device)
        r_torch = torch.linspace(r_min, r_max, N_r, dtype=torch.cfloat, device=device)
        S_H_delayed_raw, S_H_raw, S2_over_nps_raw = an.precompute_constants(S, H_unit_raw, w, t_torch, nps_raw)
        mu_raw, sigma_raw =  an.compute_mu_sigma(f1_raw, f2_raw, S_H_delayed_raw, r_torch, S_H_raw, S2_over_nps_raw, signal_amp,pulse_center_ratio=0.5, use_interp=True)
        np.save(f"{BASE}/outputs/mu_sigmas/channel{channel}_mu_sigma_{save_name}.npy",[mu_raw.cpu().numpy(),sigma_raw.cpu().numpy()])
